# 第7章：基于 SpMV 数据流的 HCCL 多设备通信实验

> 实验主入口：[HCCL + Ascend C 分布式 SpMV 实验手册](EXPERIMENT_GUIDE.md)。报告必须同时核对 Ascend C local SpMV 与 ACL+HCCL 通信的真实后端证据。

本章从 OpenMP 单机多核和 MPI 多进程继续前进到 HCCL。工程使用 ACL Device buffer、HCCL Broadcast/AllGather 和 Ascend C RTC row-block SpMV Kernel；CPU SpMV 仅生成正确性 reference。

## 前置要求

- 完成 MPI、ACL 章节
- 理解 rank、CSR 行划分和集合通信
- 目标环境具有多个可用 Ascend Device 和 HCCL

## 本章学习目标

- 初始化 rank/device/communicator
- 解释 padding、Broadcast 和 AllGather
- 分析通信、transfer、NPU compute 和端到端时间

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 章节内容

- [07.01_chapter_intro](07.01_chapter_intro.ipynb)：技术演进和能力边界
- [07.02_hccl_rank_and_communicator](07.02_hccl_rank_and_communicator.ipynb)：rank/device/communicator
- [07.03_distributed_spmv_dataflow](07.03_distributed_spmv_dataflow.ipynb)：Broadcast/AllGather 数据流
- [07.04_hccl_spmv_implementation](07.04_hccl_spmv_implementation.ipynb)：构建与运行
- [07.05_multi_npu_scaling_analysis](07.05_multi_npu_scaling_analysis.ipynb)：多 rank 扩展性
- [07.06_chapter_test](07.06_chapter_test.ipynb)：通信单变量实验

## 实验材料说明

本章完整工程位于当前章节的 `src/`，练习参考位于 `answer/`。Notebook 使用相对路径访问材料，不依赖开发者本机的原始工程位置。

## 预期现象与结果分析

上面的目录检查应显示本章 Notebook、`answer`、`images` 和 `src`。若文件缺失，应先检查课程检出是否完整，而不是继续执行后续实验。按目录顺序学习，先确认正确性，再记录性能；不要用未启用真实后端的 stub 或 reference 路径代表 NPU 性能。

## 章节小结

本节给出了本章的能力目标、材料入口和学习顺序。下一节开始进入具体知识与实验。

## 实验工程说明与本章任务

本章实验工程位于 `src/hccl_spmv/`。`hccl_context.cpp` 建立 ACL/HCCL 上下文；`spmv_distributed_hccl.cpp` 完成 Broadcast/AllGather；`main.cpp` 管理 rank/device；`generate_rank_table.py`、`run.sh`、`run_scaling.sh` 负责配置和运行。

### 本章实验任务

生成 rank table → 确认 real backend → 建 communicator → 运行 Broadcast/AllGather → 改 2/4/8 rank → 查 rank 日志 → 记录通信/端到端时间；local SpMV 必须由 Ascend C RTC Kernel 执行。

所有路径均相对 Notebook 当前目录。先检查环境，再运行真实工程；历史结果只用于观察趋势。

### 完成标准

能够指出核心源码和脚本职责，完成可用环境内的构建/诊断，并按“Ranks、HCCL、Transfer、Local NPU SpMV、Total、Error、日志状态”记录证据。
